G
# DS2002 — SQL Challenge Set (Homework)
## Music Streaming Service Database

**Due:** See Canvas  
**Submission:** GitHub

---

In this assignment, you will design and query a relational database for a **music streaming service**.
You will create tables, define primary and foreign keys, insert sample data, and answer analytical questions using SQL.

This homework builds directly on what you learned in:
- SQL Fundamentals in Notebooks (Lecture)
- SQLite + Joins + Grouping (Studio)

Read carefully. This is not a copy-paste exercise.



## Ground Rules (Read First)

- This is a **Kaggle notebook**
- Python cells must contain **valid Python**
- SQL must be executed using the helper function `q(""" SQL HERE """)`
- Do **not** paste raw SQL into a Python cell
- Do **not** use Markdown SQL fences in code cells

If you see `TODO`, you are expected to replace it.



## Part 0 — Setup (Run This Cell)

This cell creates an in-memory SQLite database and helper functions.


In [123]:

import sqlite3
import pandas as pd
from IPython.display import display, Markdown

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

def exec_sql(sql: str):
    cur.executescript(sql)
    conn.commit()

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

def run_or_todo(sql: str, label: str):
    if "TODO" in sql:
        print(f"{label}: TODO — write the SQL, then re-run this cell.")
        return None
    df = q(sql)
    display(df)
    return df

print("SQLite ready.")


SQLite ready.



## Scenario: Music Streaming Service

You are designing the backend database for a music streaming service.

Users listen to **tracks**, not albums.
Albums group tracks.
Artists release albums.

Hierarchy:
**Artist → Album → Track**
Listening happens at the **Track** level.



## Part 1 — Database Design (DDL)

You must create the following tables:

- `users`
- `artists`
- `albums`
- `tracks`
- `listens`

### Relationship Requirements
- One artist → many albums
- One album → many tracks
- One user ↔ many tracks (via listens)

Create tables with:
- Primary keys
- Foreign keys
- Reasonable data types

Replace all TODOs below.


In [124]:
exec_sql('''
DROP TABLE IF EXISTS listens;
DROP TABLE IF EXISTS tracks;
DROP TABLE IF EXISTS albums;
DROP TABLE IF EXISTS artists;
DROP TABLE IF EXISTS users;

CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE,
    email TEXT NOT NULL UNIQUE
);

CREATE TABLE artists (
    artist_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE,
    country TEXT
);

CREATE TABLE albums (
    album_id INTEGER PRIMARY KEY,
    artist_id INTEGER NOT NULL,
    title TEXT NOT NULL,
    release_year INTEGER,
    FOREIGN KEY (artist_id) REFERENCES artists(artist_id)
        ON DELETE CASCADE
        ON UPDATE CASCADE,
    UNIQUE (artist_id, title)
);

CREATE TABLE tracks (
    track_id INTEGER PRIMARY KEY,
    album_id INTEGER NOT NULL,
    title TEXT NOT NULL,
    track_number INTEGER NOT NULL,
    duration_seconds INTEGER NOT NULL CHECK (duration_seconds > 0),
    FOREIGN KEY (album_id) REFERENCES albums(album_id)
        ON DELETE CASCADE
        ON UPDATE CASCADE,
    UNIQUE (album_id, track_number),
    UNIQUE (album_id, title)
);

CREATE TABLE listens (
    listen_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    track_id INTEGER NOT NULL,
    listened_at TEXT NOT NULL DEFAULT (datetime('now')),
    seconds_listened INTEGER NOT NULL CHECK (seconds_listened >= 0),
    FOREIGN KEY (user_id) REFERENCES users(user_id)
        ON DELETE CASCADE
        ON UPDATE CASCADE,
    FOREIGN KEY (track_id) REFERENCES tracks(track_id)
        ON DELETE CASCADE
        ON UPDATE CASCADE
);
''')

print("Tables created.")

Tables created.



## Part 2 — Insert Sample Data (DML)

Insert **at minimum**:
- 5 users
- 4 artists
- 6 albums
- 15 tracks
- 30 listens

Data should be realistic enough that aggregations are meaningful.


In [125]:
exec_sql('''
DELETE FROM listens;
DELETE FROM tracks;
DELETE FROM albums;
DELETE FROM artists;
DELETE FROM users;

INSERT INTO users (user_id, name, email) VALUES
(1, 'nate', 'nate@example.com'),
(2, 'sophia', 'sophia@example.com'),
(3, 'key', 'key@example.com'),
(4, 'liam', 'liam@example.com'),
(5, 'ava', 'ava@example.com');

INSERT INTO artists (artist_id, name, country) VALUES
(1, 'Fred again..', 'UK'),
(2, 'Sam Barber', 'US'),
(3, 'Radiohead', 'UK'),
(4, 'The Lumineers', 'US');

INSERT INTO albums (album_id, artist_id, title, release_year) VALUES
(1, 1, 'Actual Life 3 (January 1 - September 9 2022)', 2022),
(2, 2, 'Restless Mind', 2024),
(3, 3, 'OK Computer', 1997),
(4, 4, 'Cleopatra', 2016);

INSERT INTO tracks (track_id, album_id, title, track_number, duration_seconds) VALUES
(1, 1, 'January 1st 2022', 1, 212),
(2, 1, 'Delilah (pull me out of this)', 2, 250),
(3, 1, 'Danielle (smile on my face)', 3, 213),
(4, 2, 'Man You Raised', 1, 215),
(5, 2, 'Cold, Dark Place', 2, 207),
(6, 2, 'Straight and Narrow', 3, 235),
(7, 3, 'Paranoid Android', 1, 388),
(8, 3, 'Karma Police', 2, 262),
(9, 3, 'No Surprises', 3, 228),
(10, 4, 'Sleep on the Floor', 1, 211),
(11, 4, 'Ophelia', 2, 160),
(12, 4, 'Angela', 3, 201);

INSERT INTO listens (listen_id, user_id, track_id, listened_at, seconds_listened) VALUES
(1, 1, 8, '2026-01-17 08:10:00', 262),
(2, 2, 11, '2026-01-17 09:30:00', 160),
(3, 3, 2, '2026-01-17 10:05:00', 250),
(4, 4, 7, '2026-01-17 12:40:00', 310),
(5, 2, 10, '2026-01-17 14:15:00', 211),
(6, 1, 3, '2026-01-17 16:55:00', 213),
(7, 2, 9, '2026-01-18 07:05:00', 228),
(8, 3, 6, '2026-01-18 08:20:00', 235),
(9, 4, 11, '2026-01-18 09:50:00', 160),
(10, 3, 2, '2026-01-18 11:10:00', 200),
(11, 1, 12, '2026-01-18 13:25:00', 201),
(12, 2, 5, '2026-01-18 21:45:00', 150),
(13, 3, 8, '2026-01-19 07:30:00', 262),
(14, 3, 8, '2026-01-19 08:05:00', 180),
(15, 4, 8, '2026-01-19 09:40:00', 240),
(16, 4, 8, '2026-01-19 10:15:00', 262),
(17, 1, 8, '2026-01-19 11:50:00', 210),
(18, 2, 8, '2026-01-19 22:20:00', 262),
(19, 1, 11, '2026-01-20 07:10:00', 160),
(20, 2, 11, '2026-01-20 07:35:00', 160),
(21, 3, 11, '2026-01-20 08:05:00', 120),
(22, 4, 11, '2026-01-20 08:40:00', 160),
(23, 1, 11, '2026-01-20 09:15:00', 160),
(24, 1, 11, '2026-01-20 21:55:00', 100),
(25, 4, 7, '2026-01-21 06:45:00', 388),
(26, 4, 7, '2026-01-21 07:55:00', 260),
(27, 2, 7, '2026-01-21 12:10:00', 300),
(28, 1, 7, '2026-01-21 18:30:00', 388),
(29, 2, 7, '2026-01-22 09:10:00', 200),
(30, 3, 7, '2026-01-23 23:45:00', 388);
''')

print("Sample data inserted.")

Sample data inserted.



## Part 3 — SQL Challenge Questions

Write SQL queries to answer each question.
Each cell should return a table.


### Q1. List all users.

In [126]:

run_or_todo('''
SELECT name
FROM users
''', label="Q1");


,name
0,ava
1,key
2,liam
3,nate
4,sophia


### Q2. List all artists.

In [127]:

run_or_todo('''
SELECT name
FROM artists
''', label="Q2");


,name
0,Fred again..
1,Radiohead
2,Sam Barber
3,The Lumineers


### Q3. List all albums with their artist name.

In [128]:

run_or_todo('''
SELECT al.title AS album_title, ar.name AS artist_name
FROM albums al
JOIN artists ar ON al.artist_id = ar.artist_id
''', label="Q3");


,album_title,artist_name
0,Actual Life 3 (January 1 - September 9 2022),Fred again..
1,Restless Mind,Sam Barber
2,OK Computer,Radiohead
3,Cleopatra,The Lumineers


### Q4. List all tracks with their album title and artist name.

In [129]:

run_or_todo('''
SELECT t.title AS track, al.title AS album, ar.name AS artist
FROM tracks t
JOIN albums al ON al.album_id = t.album_id
JOIN artists ar ON al.artist_id = ar.artist_id
''', label="Q4");


,track,album,artist
0,Danielle (smile on my face),Actual Life 3 (January 1 - September 9 2022),Fred again..
1,Delilah (pull me out of this),Actual Life 3 (January 1 - September 9 2022),Fred again..
2,January 1st 2022,Actual Life 3 (January 1 - September 9 2022),Fred again..
3,"Cold, Dark Place",Restless Mind,Sam Barber
4,Man You Raised,Restless Mind,Sam Barber
5,Straight and Narrow,Restless Mind,Sam Barber
6,Karma Police,OK Computer,Radiohead
7,No Surprises,OK Computer,Radiohead
8,Paranoid Android,OK Computer,Radiohead
9,Angela,Cleopatra,The Lumineers


### Q5. Show all listening events with user name, track title, and timestamp.

In [130]:
run_or_todo('''
SELECT u.name AS user_name, t.title AS track_title, l.listened_at
FROM listens l
JOIN users u ON l.user_id = u.user_id
JOIN tracks t ON l.track_id = t.track_id
ORDER BY l.listened_at
''', label="Q5");

,user_name,track_title,listened_at
0,nate,Karma Police,2026-01-17 08:10:00
1,sophia,Ophelia,2026-01-17 09:30:00
2,key,Delilah (pull me out of this),2026-01-17 10:05:00
3,liam,Paranoid Android,2026-01-17 12:40:00
4,sophia,Sleep on the Floor,2026-01-17 14:15:00
5,nate,Danielle (smile on my face),2026-01-17 16:55:00
6,sophia,No Surprises,2026-01-18 07:05:00
7,key,Straight and Narrow,2026-01-18 08:20:00
8,liam,Ophelia,2026-01-18 09:50:00
9,key,Delilah (pull me out of this),2026-01-18 11:10:00


### Q6. How many tracks does each album contain?

In [131]:
run_or_todo('''
SELECT al.title AS album_title, ar.name AS artist_name, COUNT(t.track_id) AS num_tracks
FROM albums al
JOIN artists ar ON al.artist_id = ar.artist_id
LEFT JOIN tracks t ON al.album_id = t.album_id
GROUP BY al.album_id, al.title, ar.name
''', label="Q6");

,album_title,artist_name,num_tracks
0,Actual Life 3 (January 1 - September 9 2022),Fred again..,3
1,Restless Mind,Sam Barber,3
2,OK Computer,Radiohead,3
3,Cleopatra,The Lumineers,3


### Q7. How many listens does each track have?

In [132]:
run_or_todo('''
SELECT t.title AS track_title, ar.name AS artist_name, COUNT(l.listen_id) AS listen_count
FROM tracks t
JOIN albums al ON t.album_id = al.album_id
JOIN artists ar ON al.artist_id = ar.artist_id
LEFT JOIN listens l ON t.track_id = l.track_id
GROUP BY t.track_id, t.title, ar.name
''', label="Q7");

,track_title,artist_name,listen_count
0,January 1st 2022,Fred again..,0
1,Delilah (pull me out of this),Fred again..,2
2,Danielle (smile on my face),Fred again..,1
3,Man You Raised,Sam Barber,0
4,"Cold, Dark Place",Sam Barber,1
5,Straight and Narrow,Sam Barber,1
6,Paranoid Android,Radiohead,7
7,Karma Police,Radiohead,7
8,No Surprises,Radiohead,1
9,Sleep on the Floor,The Lumineers,1


### Q8. How many listens has each user made?

In [133]:
run_or_todo('''
SELECT u.name AS user_name, COUNT(l.listen_id) AS listen_count
FROM users u
LEFT JOIN listens l ON u.user_id = l.user_id
GROUP BY u.user_id, u.name
''', label="Q8");

,user_name,listen_count
0,ava,0
1,key,7
2,liam,7
3,nate,8
4,sophia,8


### Q9. Which tracks have been listened to more than 3 times?

In [134]:
run_or_todo('''
SELECT t.title AS track_title, COUNT(l.listen_id) AS listen_count
FROM tracks t
JOIN listens l ON t.track_id = l.track_id
GROUP BY t.track_id, t.title
HAVING COUNT(l.listen_id) > 3
ORDER BY listen_count DESC, track_title
''', label="Q9");

,track_title,listen_count
0,Ophelia,8
1,Karma Police,7
2,Paranoid Android,7


### Q10. Which album has the most total listens?

In [135]:
run_or_todo('''
SELECT al.title AS album_title, ar.name AS artist_name, COUNT(l.listen_id) AS total_listens
FROM albums al
JOIN artists ar ON al.artist_id = ar.artist_id
JOIN tracks t ON al.album_id = t.album_id
JOIN listens l ON t.track_id = l.track_id
GROUP BY al.album_id, al.title, ar.name
ORDER BY total_listens DESC
LIMIT 1
''', label="Q10");

,album_title,artist_name,total_listens
0,OK Computer,Radiohead,15


### Q11. Which artist has the most total listens?

In [136]:
run_or_todo('''
SELECT ar.name AS artist_name, COUNT(l.listen_id) AS total_listens
FROM artists ar
JOIN albums al ON ar.artist_id = al.artist_id
JOIN tracks t ON al.album_id = t.album_id
JOIN listens l ON t.track_id = l.track_id
GROUP BY ar.artist_id, ar.name
ORDER BY total_listens DESC
LIMIT 1
''', label="Q11");

,artist_name,total_listens
0,Radiohead,15


### Q12. What is the average number of listens per user?

In [137]:
run_or_todo('''
SELECT AVG(listen_count) AS avg_listens_per_user
FROM (
    SELECT u.user_id, COUNT(l.listen_id) AS listen_count
    FROM users u
    LEFT JOIN listens l ON u.user_id = l.user_id
    GROUP BY u.user_id
)
''', label="Q12");

,avg_listens_per_user
0,6.0


### Q13. Which users have never listened to a track?

In [138]:
run_or_todo('''
SELECT u.name AS user_name
FROM users u
LEFT JOIN listens l ON u.user_id = l.user_id
WHERE l.listen_id IS NULL
ORDER BY u.name
''', label="Q13");

,user_name
0,ava


### Q14. Show the top 5 most-listened-to tracks.

In [139]:
run_or_todo('''
SELECT t.title AS track_title, ar.name AS artist_name, COUNT(l.listen_id) AS listen_count
FROM tracks t
JOIN albums al ON t.album_id = al.album_id
JOIN artists ar ON al.artist_id = ar.artist_id
JOIN listens l ON t.track_id = l.track_id
GROUP BY t.track_id, t.title, ar.name
ORDER BY listen_count DESC, track_title
LIMIT 5
''', label="Q14");

,track_title,artist_name,listen_count
0,Ophelia,The Lumineers,8
1,Karma Police,Radiohead,7
2,Paranoid Android,Radiohead,7
3,Delilah (pull me out of this),Fred again..,2
4,Angela,The Lumineers,1


### Q15. Explain why the listens table cannot be merged into users or tracks.

In [140]:
run_or_todo('''
SELECT 'If merged into users or tracks, it would prevent recording multiple listens per user per track over time.' AS explanation
''', label="Q15");

,explanation
0,"If merged into users or tracks, it would preve..."



## Part 4 — Reflection

Answer in 3–5 sentences.


In [141]:

reflection = """
 What was easiest? It was actually pretty easy, I thought the lab was harder.
 What was hardest? Maybe inserting data and creating the tables
 What did this assignment help you understand? nothing really, maybe creating tables
"""

print(reflection)



 What was easiest? It was actually pretty easy, I thought the lab was harder.
 What was hardest? Maybe inserting data and creating the tables
 What did this assignment help you understand? nothing really, maybe creating tables




## Grading Rubric (100 points)

### Database Design (30 pts)
- Correct tables created (10)
- Correct primary keys (10)
- Correct foreign keys & relationships (10)

### Data Insertion (15 pts)
- Meets minimum data requirements
- Referential integrity preserved

### SQL Queries (40 pts)
- Correct logic & joins (25)
- Proper GROUP BY / HAVING usage (10)
- Clean, readable queries (5)

### Reflection & Notebook Quality (15 pts)
- Reflection completed (5)
- All cells run without error (10)



## Submission Checklist

- All TODOs replaced
- All cells run successfully
- Outputs visible
- Notebook saved
- Git URL submitted in Canvas
